In [ ]:
# Jupyter notebook to test passband_ripple implementation and plot its output

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

In [ ]:
def plot_filter_response(num_taps=2, max_ripple_db=1.0, decay_rate=0.1, seed=None):
    rng = np.random.default_rng(seed=seed)
    
    
    # weights = generate_ripple_filter(
    #     num_taps=num_taps, 
    #     max_ripple_db=max_ripple_db, 
    #     coefficient_decay_rate=decay_rate, 
    #     rng=rng
    # )
    weights = prototype_base_filter(
        ripple_db=max_ripple_db,
        rng=rng
    )


    # Compute frequency response
    w, h = signal.freqz(weights, worN=2048)
    
    # Convert w (rad/sample) to normalized frequency (cycles/sample)
    freq = w / (2 * np.pi)
    mag_db = 20 * np.log10(np.abs(h) + 1e-12)
    phase = np.unwrap(np.angle(h))
    
    plt.figure(figsize=(12, 8))
    
    # Magnitude Response
    plt.subplot(2, 1, 1)
    plt.plot(freq, mag_db)
    plt.title(f"Filter Frequency Response (taps={num_taps}, max_ripple={max_ripple_db}dB, decay={decay_rate})")
    plt.ylabel("Magnitude (dB)")
    plt.grid(True)
    plt.ylim(np.min(mag_db) - 2, np.max(mag_db) + 2)
    
    # Phase Response
    plt.subplot(2, 1, 2)
    plt.plot(freq, phase)
    plt.ylabel("Phase (radians)")
    plt.xlabel("Normalized Frequency (cycles/sample)")
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig("passband_ripple_response.png")
    print("Plot saved as passband_ripple_response.png")
    plt.show()

In [ ]:
def generate_ripple_filter(num_taps=2, max_ripple_db=2.0, coefficient_decay_rate=1.0, 
                           rng=None, max_counter=1000, fft_len=1024):
    """
    Re-implementation of the filter generation logic from torchsig.transforms.functional.passband_ripple
    to allow access to the filter coefficients for plotting.
    """
    rng = rng or np.random.default_rng()
    eps = 1e-12
    counter = 0
    estimate_ripple_db = max_ripple_db + 1.0
    #max_ripple_lin = 10^(max_ripple_db/10)
    decay = np.exp(-coefficient_decay_rate * np.arange(num_taps))
    
    while estimate_ripple_db > max_ripple_db and counter < max_counter:
        # Draw random complex Gaussian taps
        gaussian = rng.normal(0, 1, num_taps) + 1j * rng.normal(0, 1, num_taps)
        weights = gaussian * decay
        max_abs = np.max(np.abs(weights))
        if max_abs == 0:
            counter += 1
            continue
        weights /= max_abs
        
        # Measure ripple across the entire band
        fft_raw = np.fft.fft(weights, fft_len)
        fft_mag = np.abs(fft_raw)
        # Shift to center frequency for ripple measurement (as done in passband_ripple)
        fft_mag = np.roll(fft_mag, fft_len // 2)
        fft_mag += eps
        fft_db = 20 * np.log10(fft_mag)
        estimate_ripple_db = np.ptp(fft_db) # peak-to-peak
        counter += 1
        
    if estimate_ripple_db > max_ripple_db:
        print(f"Warning: Could not satisfy ripple spec after {max_counter} tries. "
              f"Best obtained: {estimate_ripple_db:.2f} dB")
        
    print(counter)
    return weights

In [ ]:
def prototype_base_filter(ripple_db=2.0, coefficient_decay_rate=0.05, fft_len=1024, rng=None):
    """
    Creates a fixed 2-tap prototype filter with the specified ripple, then 
    randomizes the positioning (center, flip). Maintains final acceptance check.
    """
    rng = rng or np.random.default_rng()
    ripple_linear =  10 ** (ripple_db / 2 / 20)

    # prototype 2-tap filter with specified ripple_db
    weights = [1 + 1j, (1-ripple_linear) + (1-ripple_linear)*1j  ]
    decay = np.exp(-coefficient_decay_rate * np.arange(2))
    weights = weights * decay
    
    # scale
    max_abs = np.max(np.abs(weights))
    weights /= max_abs
    
    # Acceptance test: measure ripple across the entire band
    fft_raw = np.fft.fft(weights, fft_len)
    fft_mag = np.abs(fft_raw)
    fft_db = 20 * np.log10(fft_mag)
    estimate_ripple_db = np.ptp(fft_db) # peak-to-peak    
    print('estimate_ripple_db: ', estimate_ripple_db)
    
    if np.abs(estimate_ripple_db) > 1.5*np.abs(ripple_db) :
        print(f"Warning: Did not satisfy ripple spec. ")

    return weights

In [ ]:
plot_filter_response(num_taps=2, max_ripple_db=0.75, decay_rate=0.05, seed=43)